# Task 3.2: Failure Mode Analysis

### Expected Failure Scenario
The method uses a low-degree mathematically explicit polynomial to try and emulate the complex dimensional space achieved naturally by RBF kernel SVMs. I expect this method to fail catastrophically on the **Interlocked Moons** dataset (`make_moons`). 
The structural condition here is that navigating between two interwoven semi-circular arcs requires an "S-curve" boundary. The paper fundamentally relies on `Assumption 2`: that a simple low-degree polynomial mapping is adequately sufficient to capture the non-linear boundaries of real-world patterns. However, a pure degree-2 $O(x_1^2, x_2^2)$ polynomial maps exclusively to simple *conic sections* (like single circles, ellipses, or linear hyperbolas). This fixed low-degree math lacks the topological complexity needed to intertwine through complex multi-curvature spaces.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Generate interlocking moons (highly non-linear, non-conic)
X_m, y_m = make_moons(n_samples=500, noise=0.15, random_state=42)
X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

# Our Explicit Polynomial Method (Degree-2)
poly_m = PolynomialFeatures(degree=2, include_bias=False)
clf_poly_moon = LinearSVC(C=1.0, random_state=42, dual=False)
clf_poly_moon.fit(poly_m.fit_transform(X_m_tr), y_m_tr)
acc_poly_moon = accuracy_score(y_m_te, clf_poly_moon.predict(poly_m.transform(X_m_te)))

# Baseline RBF SVM (Infinite dimensionality capabilities)
clf_rbf_moon = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
clf_rbf_moon.fit(X_m_tr, y_m_tr)
acc_rbf_moon = accuracy_score(y_m_te, clf_rbf_moon.predict(X_m_te))

print(f'Failure Target (Explicit Poly d=2) Accuracy: {acc_poly_moon*100:.2f}%')
print(f'RBF Competitor Accuracy: {acc_rbf_moon*100:.2f}%')

os.makedirs('/Users/belalraza/Desktop/adm/partB/results', exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 100), np.linspace(-1.0, 1.5, 100))

# Plot explicitly polynomial limit
Z_pm = clf_poly_moon.predict(poly_m.transform(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
ax[0].contourf(xx, yy, Z_pm, alpha=0.3, cmap='bwr')
ax[0].scatter(X_m_te[:, 0], X_m_te[:, 1], c=y_m_te, edgecolors='k', cmap='bwr')
ax[0].set_title(f'Explicit Poly (d=2) Boundary - Failed ({acc_poly_moon*100:.1f}%)')

# Plot infinite dimensional RBF success
Z_rbf = clf_rbf_moon.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[1].contourf(xx, yy, Z_rbf, alpha=0.3, cmap='bwr')
ax[1].scatter(X_m_te[:, 0], X_m_te[:, 1], c=y_m_te, edgecolors='k', cmap='bwr')
ax[1].set_title(f'RBF SVM Boundary ({acc_rbf_moon*100:.1f}%)')
plt.savefig('/Users/belalraza/Desktop/adm/partB/results/failure_mode.png')


### Why the Method Fails
The method fails heavily in this scenario because it violates the critical design assumption previously identified in **Question 1 (Assumption 2)**—namely, that a low-degree mapping $d=2$ inherently captures the necessary spatial interactions of the underlying data distribution. As distinctly visible in the plotted failure, the degree-2 polynomial equation completely fails because it is mathematically locked to drawing continuous standard conic geometries (here representing a hyperbola slice), effectively "underfitting" any complex shape requiring sinusoidal or "S" curve interactions. The RBF pipeline, expanding to infinite dimensions, successfully tracks the gap accurately.

**Suggested Modification:** A concrete modification to address this dimensional ceiling is to dynamically test and escalate to $d=3$ (generating cubic interaction functions) selectively for subsets of heavily entangled training regions, expanding the explicit capacity of the linear mapping without paying the $O(n^4)$ cost across identically uniform spaces.